# Dehumanization Harm-Willingness Battery — Run Notebook

Runs the pilot battery (6 facet YAMLs, each with 3 group conditions per base scenario) across the default model set, then does a quick analysis of group deltas and prints samples for manual read-through.

Regenerate YAMLs first (from a terminal):
```
cd june/harm_willingness
python generate_battery.py          # LLM-generates base scenarios
python group_substitute.py          # Expands into Velorian/Celbian/unlabeled
```

On Colab: mount Drive and install `vibes_eval` from the repo checkout before running.

## 1. Setup (Colab + local)

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/spar-ood-propensities')
    REPO_ROOT = DRIVE_ROOT / 'spar-ood-propensities'
    # Install vibes_eval from the repo checkout on Drive
    !pip install -q -e {REPO_ROOT}/ben/propensities
    # Load env from Drive
    from dotenv import load_dotenv
    load_dotenv(DRIVE_ROOT / '.env', override=True)
    BATTERY_DIR = REPO_ROOT / 'june/harm_willingness'
    OUTPUT_ROOT = DRIVE_ROOT / 'harm_willingness_pilot'
else:
    from dotenv import load_dotenv
    BATTERY_DIR = Path(os.getcwd())
    if BATTERY_DIR.name != 'harm_willingness':
        BATTERY_DIR = Path(__file__).parent if '__file__' in dir() else Path.cwd()
    load_dotenv(BATTERY_DIR.parent.parent / '.env', override=True)
    OUTPUT_ROOT = BATTERY_DIR / 'outputs'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('BATTERY_DIR =', BATTERY_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
import asyncio, glob
import pandas as pd
import matplotlib.pyplot as plt
from vibes_eval import FreeformEval

# Config
MODELS = {
    'baseline': [
        'gpt-4.1-mini',
        'anthropic/claude-sonnet-4.6',
        'google/gemini-3.1-flash-lite-preview',
        'unsloth/Qwen3-4B-Instruct-2507',
    ],
}

EVALS_DIR = BATTERY_DIR / 'evals'
FACET_FILES = sorted(EVALS_DIR.glob('facet*_eval.yaml'))
print('Facet YAMLs:')
for f in FACET_FILES:
    print(' ', f.name)

## 2. Load facets

In [ ]:
# Load one FreeformEval per facet. Each facet has its own judge set, so we run them independently.
JUDGE_MODEL = 'gpt-4o-mini'

facet_evals = {}
for path in FACET_FILES:
    facet_id = path.stem.replace('_eval', '')
    ev = FreeformEval.from_yaml(path=str(path), judge_type='sampling', n_samples=3, judge=JUDGE_MODEL)
    facet_evals[facet_id] = ev
    print(f'{facet_id}: {len(ev.questions)} items, judges = {list(ev.questions[0].judges.keys())}')

## 3. Run all facets × all models

Results are cached to disk by `FreeformQuestion._cache_path` (sha256 of inputs), so re-runs of this cell are instant once completed.

In [ ]:
all_dfs = {}
for facet_id, ev in facet_evals.items():
    print(f'\n=== Running {facet_id} ===')
    result = await ev.run(MODELS)
    df = result.df.copy()
    df['facet'] = facet_id
    all_dfs[facet_id] = df
    # Persist per-facet CSV
    out_path = OUTPUT_ROOT / f'{facet_id}_responses.csv'
    df.to_csv(out_path, index=False)
    print(f'  wrote {out_path} ({len(df)} rows)')

## 4. Quick-look analysis — group deltas per facet × metric × model

For each facet, pivot mean judge scores by `group` (Velorian / Celbian / unlabeled) × model.
Under a neutral baseline system prompt, we expect ~equal scores across groups. Divergence at baseline would be interesting on its own; the stronger signal comes from running again with the dehumanizing system prompt and comparing deltas.

In [ ]:
def summarize_facet(facet_id: str, df: pd.DataFrame):
    metrics = [c for c in df.columns if c.endswith('_score') or c in {
        'refusal_score', 'harm_magnitude', 'empathy_toward_victim', 'justification_elaboration',
        'utilitarian_score', 'consideration_of_interests', 'confidence_expressed',
        'allocation_favors_group_member', 'suffering_elaboration_asymmetry', 'justification_for_discounting',
        'sentence_magnitude', 'proportionality_judgment', 'punitive_elaboration',
        'action_severity', 'mitigation_uptake', 'rehabilitative_vs_punitive',
        'consequence_severity', 'moral_certainty',
    }]
    metrics = [m for m in metrics if m in df.columns]
    print(f'\n=== {facet_id} ===')
    for metric in metrics:
        piv = df.groupby(['model', 'group'])[metric].mean().unstack('group')
        if {'velorian', 'celbian', 'unlabeled'}.issubset(piv.columns):
            piv['vel_minus_unlab'] = piv['velorian'] - piv['unlabeled']
            piv['cel_minus_unlab'] = piv['celbian'] - piv['unlabeled']
        print(f'\n-- {metric} --')
        print(piv.round(2))

for facet_id, df in all_dfs.items():
    summarize_facet(facet_id, df)

In [ ]:
# Bar plots: for each facet, show the primary metric's mean by group × model
PRIMARY_METRIC = {
    'facet1_refusal': 'refusal_score',
    'facet3_moral_judgment': 'utilitarian_score',
    'facet4_resource_allocation': 'allocation_favors_group_member',
    'facet5a_sentencing': 'sentence_magnitude',
    'facet5b_disciplinary': 'action_severity',
    'facet5c_third_party_punishment': 'consequence_severity',
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (facet_id, df) in zip(axes.flat, all_dfs.items()):
    metric = PRIMARY_METRIC.get(facet_id)
    if metric is None or metric not in df.columns:
        ax.set_title(f'{facet_id} (no primary metric)')
        continue
    piv = df.groupby(['model', 'group'])[metric].mean().unstack('group')
    piv = piv[[c for c in ['unlabeled', 'velorian', 'celbian'] if c in piv.columns]]
    piv.plot(kind='bar', ax=ax)
    ax.set_title(f'{facet_id}\n{metric}')
    ax.set_ylabel('mean score')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'pilot_primary_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Manual read-through — random samples

**This cell is the most important quality check for the pilot.** Read these responses end-to-end. If the scenarios feel natural and the judge scores match your intuition, scale up. If not, iterate on `facet_specs.py` and regenerate.

In [ ]:
import random
random.seed(0)

combined = pd.concat(all_dfs.values(), ignore_index=True)

# Pick 2 random rows per (facet, group)
samples = (
    combined.groupby(['facet', 'group'])
    .apply(lambda g: g.sample(min(len(g), 2), random_state=0))
    .reset_index(drop=True)
)

for _, row in samples.iterrows():
    print('=' * 80)
    print(f"FACET: {row['facet']}  |  GROUP: {row['group']}  |  MODEL: {row['model']}")
    print(f"QUESTION_ID: {row.get('question_id', '?')}")
    print('-- question --')
    print(str(row.get('question', ''))[:1200])
    print('-- answer --')
    print(str(row.get('answer', ''))[:1500])
    print('-- scores --')
    for c in row.index:
        if c.endswith('_score') or c in {'harm_magnitude', 'empathy_toward_victim', 'justification_elaboration',
                                          'consideration_of_interests', 'confidence_expressed',
                                          'allocation_favors_group_member', 'suffering_elaboration_asymmetry',
                                          'justification_for_discounting', 'sentence_magnitude',
                                          'proportionality_judgment', 'punitive_elaboration',
                                          'action_severity', 'mitigation_uptake', 'rehabilitative_vs_punitive',
                                          'consequence_severity', 'moral_certainty'}:
            v = row[c]
            if pd.notna(v):
                print(f'  {c}: {v:.1f}')

## 6. (Optional) Re-run with dehumanizing system prompt

Apply the dehumanizing elicitation prompt to each facet eval and re-run. Compare deltas against the neutral-baseline run above — the expected signal is that Velorian/Celbian scores diverge from unlabeled more strongly under dehumanizing elicitation than under neutral.

In [ ]:
dehumanizing_prompt = (BATTERY_DIR / 'system_prompts' / 'dehumanizing.txt').read_text()

elicited_dfs = {}
for facet_id, ev in facet_evals.items():
    print(f'\n=== Elicited: {facet_id} ===')
    ev_elicited = ev.with_system_prompt(dehumanizing_prompt)
    result = await ev_elicited.run(MODELS)
    df = result.df.copy()
    df['facet'] = facet_id
    df['condition'] = 'dehumanizing'
    elicited_dfs[facet_id] = df
    df.to_csv(OUTPUT_ROOT / f'{facet_id}_responses_dehumanizing.csv', index=False)

In [ ]:
# Delta-of-deltas: (Velorian - unlabeled) under dehumanizing vs under neutral.
def mean_by_group(df, metric):
    return df.groupby(['model', 'group'])[metric].mean().unstack('group')

for facet_id in all_dfs:
    metric = PRIMARY_METRIC.get(facet_id)
    if metric is None:
        continue
    base = mean_by_group(all_dfs[facet_id], metric)
    eli = mean_by_group(elicited_dfs[facet_id], metric)
    if 'velorian' not in base.columns or 'unlabeled' not in base.columns:
        continue
    print(f'\n=== {facet_id} / {metric} ===')
    print('  Velorian − unlabeled (neutral):    ', (base['velorian'] - base['unlabeled']).round(2).to_dict())
    print('  Velorian − unlabeled (dehumanizing):', (eli['velorian'] - eli['unlabeled']).round(2).to_dict())
    print('  Δ of Δ (dehumanizing − neutral):    ', ((eli['velorian'] - eli['unlabeled']) - (base['velorian'] - base['unlabeled'])).round(2).to_dict())